# Animation: dynamic minus static sea level, and the GRACE load

Left: the direct load each month (mm of water equivalent; linear colour scale within ±50 mm, logarithmic beyond). Middle: the dynamic sea-level
change minus the static (AA24) one, over the ocean.
The two differ only through the rotation, so this is the ocean pole tide of dynamic minus static pole: a degree-2, order-1 pattern, plus the ocean's
response to it. Right: the dynamic and static pole paths. Frames are the observed months.

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.animation import PillowWriter
from IPython.display import Image

# solver_dynamic and examples_dynamic_helper live one directory up, in new-code/
sys.path.append("..")

import solver_dynamic as sd
import examples_dynamic_helper as eh
import grace_data_helper as gh
import grace_plot_helper as ph

/space/dah94/rotation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#=======================================================================
# Parameters
#=======================================================================

# The run to plot (see grace_rotation_run.py): forcing, degree and quality factor
forcing = "land-ocean"
lmax = 64
quality_factor = "100"

# The load's colour scale is linear within +-load_linear_mm and logarithmic beyond
load_linear_mm = 50.0

frames_per_second = 8
dpi = 80
gif_path = f"animations/grace_sea_level_minus_static_{forcing}.gif"

In [3]:
#=======================================================================
# Earth model, the run and the direct load
#=======================================================================

state = sd.make_state(lmax)
model = state.model
constants = sd.rotation_constants(model)

run = gh.load_run(forcing, lmax, quality_factor)
time_days, dates, load_coeffs = gh.direct_load_coefficients(forcing, lmax)
assert np.allclose(run["month_days"], time_days), "Plot a run that covers every month"

dynamic_mas = sd.omega_to_mas(run["dynamic_omega"], constants)
static_mas = sd.omega_to_mas(run["static_omega"], constants)

# The dynamic pole at every time step, and the step of each month (the frames)
path_mas = run["pole_mas"]
frames = ph.frame_index(run["t_days"], run["month_days"])
print(f"{len(dates)} months, {dates[0]} to {dates[-1]}")

259 months, 2002-04-17 to 2026-07-16


In [4]:
#=======================================================================
# Load, and dynamic minus static sea level, at every month (mm)
#=======================================================================

mask = ph.ocean_mask(state)
load_mm = ph.fields_from_coefficients(model, load_coeffs)
difference_coeffs = run["dynamic_sea_level_coeffs"] - run["static_sea_level_coeffs"]
difference_mm = ph.fields_from_coefficients(model, difference_coeffs, scale=1e3, mask=mask)
print(f"max |dynamic - static| {np.nanmax(np.abs(difference_mm)):.2f} mm; "
      f"max |dynamic - static| pole {np.linalg.norm(dynamic_mas - static_mas, axis=1).max():.1f} mas")

max |dynamic - static| 5.37 mm; max |dynamic - static| pole 114.4 mas


In [5]:
#=======================================================================
# Animation
#=======================================================================

animation = ph.map_animation(
    state,
    [load_mm, difference_mm],
    ["Direct load", "Dynamic minus static sea level"],
    ["Direct load (mm water equivalent)", r"$\xi_{\rm dynamic}-\xi_{\rm static}$ (mm)"],
    ["BrBG", "RdBu_r"],
    [ph.load_norm(load_mm, linear_mm=load_linear_mm), ph.symmetric_norm(difference_mm, percentile=100.0)],
    dates,
    path_mas,
    frames,
    pole_static_mas=static_mas,
    frames_per_second=frames_per_second,
)

os.makedirs(os.path.dirname(gif_path), exist_ok=True)
animation.save(gif_path, writer=PillowWriter(fps=frames_per_second), dpi=dpi)
print(f"saved {gif_path}")
Image(url=gif_path)

saved animations/grace_sea_level_minus_static_land-ocean.gif
